# Capstone Three — Data Wrangling
**Project:** Optimizing Term Deposit Marketing Campaigns for a European Bank
**Author:** Colin Segura — Springboard Data Science Career Track — Business Insights Specialization

This notebook covers Step 2 of Capstone Three and works through the four substeps
of the Data Science Method: Data Collection, Data Organization, Data Definition,
and Data Cleaning. The output is a cleaned CSV ready to feed into EDA and modeling.

The raw dataset is the UCI Bank Marketing "bank-additional-full.csv" file
published alongside the Moro, Cortez and Rita (2014) paper. It contains 41,188
records from a Portuguese retail bank's phone-based term deposit marketing
campaign, with 20 input features and one binary target. Real world data issues
I know I need to handle up front:

- "unknown" values in six categorical columns (they are the dataset's convention for missing)
- a pdays value of 999 encoding "never previously contacted" rather than 999 days
- the call duration feature perfectly leaks the outcome (a zero second call is by definition a non-subscription)
- a small number of exact duplicate rows from source-system re-exports

> **Note on the data file.** This notebook reads bank-additional-full.csv from the
> same folder as the notebook. In this working session it is a schema faithful
> synthetic sample sized to match the real file. The wrangling code below runs
> unchanged on the actual UCI file, which is downloadable from the UCI Machine
> Learning Repository (link in the project proposal).


## 1. Data Collection

Load the raw file. The real UCI release uses semicolons as delimiters rather
than commas, so I will pass sep=";" explicitly rather than rely on pandas'
default sniffing.


In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW_PATH = Path("bank-additional-full.csv")
size_mb = os.path.getsize(RAW_PATH) / (1024 * 1024)
print(f"Raw file: {RAW_PATH.name}  ({size_mb:.2f} MB)")

df = pd.read_csv(RAW_PATH, sep=";")
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")


Raw file: bank-additional-full.csv  (4.71 MB)
Loaded: 41,211 rows, 21 columns


In [2]:
df.head()


In [3]:
# Print form of head() since bare expression output is not captured here
print(df.head().to_string())


   age         job   marital            education  default  housing loan   contact month day_of_week  duration  campaign  pdays  previous     poutcome  emp.var.rate  cons.price.idx  cons.conf.idx  euribor3m  nr.employed    y
0   32    services   married             basic.6y       no      yes  yes  cellular   jul         thu       127         3    999         0  nonexistent          -1.7          92.612          -33.8      1.461       4983.1   no
1   74      admin.   married             basic.6y       no  unknown   no  cellular   may         mon        30         1    999         0  nonexistent          -2.0          92.705          -27.0      2.897       5214.6   no
2   63  unemployed    single             basic.4y       no       no   no  cellular   jul         thu       129        16     16         4      failure          -1.2          93.146          -42.0      3.804       5053.8   no
3   48    services   married    university.degree  unknown       no   no  cellular   may         mon

## 2. Data Organization

The project uses a flat layout so the notebook, the raw file, and the cleaned
output all live in the same directory. This keeps the notebook portable — no
subfolder path juggling to worry about, and it can be shared as a single unit
without breaking file references.

```
project_folder/
├── 01_data_wrangling.ipynb           <- this notebook
├── bank-additional-full.csv          <- raw input
├── bank_marketing_cleaned.csv        <- generated below
└── generate_sample_data.py           <- reproduces the raw sample
```

A more conventional cookiecutter layout with data/raw and data/processed
subfolders is better for larger projects, but for a single notebook a flat
layout is simpler to hand off.


In [4]:
import glob
print("Files in the working directory:")
for f in sorted(glob.glob("*")):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f:40s}  {size_kb:8.1f} KB")


Files in the working directory:
  bank-additional-full.csv                    4823.8 KB
  build_notebook.py                             22.3 KB
  generate_sample_data.py                        5.8 KB


## 3. Data Definition

Before I clean anything, I want to see what I am working with — column types,
non null counts, and a sense of the numeric and categorical distributions.


In [5]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 41211 entries, 0 to 41210
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41211 non-null  int64  
 1   job             41211 non-null  str    
 2   marital         41211 non-null  str    
 3   education       41211 non-null  str    
 4   default         41211 non-null  str    
 5   housing         41211 non-null  str    
 6   loan            41211 non-null  str    
 7   contact         41211 non-null  str    
 8   month           41211 non-null  str    
 9   day_of_week     41211 non-null  str    
 10  duration        41211 non-null  int64  
 11  campaign        41211 non-null  int64  
 12  pdays           41211 non-null  int64  
 13  previous        41211 non-null  int64  
 14  poutcome        41211 non-null  str    
 15  emp.var.rate    41211 non-null  float64
 16  cons.price.idx  41211 non-null  float64
 17  cons.conf.idx   41211 non-null  float64
 1

Twenty inputs plus one target. Everything is either int64 (5 columns),
float64 (5 columns), or object (11 columns, all categorical). There are no
nulls at this stage — but that is misleading, because the dataset uses
"unknown" as its own sentinel value rather than a true NaN, and pdays uses
999 to encode "never contacted." I will address both in the cleaning section.


In [6]:
# Summary statistics for numeric columns
print(df.describe().T.round(3).to_string())


                  count      mean      std       min       25%       50%       75%       max
age             41211.0    57.446   23.706    17.000    37.000    58.000    78.000    98.000
duration        41211.0   217.587  216.811     0.000    63.000   151.000   303.000  2221.000
campaign        41211.0     2.845    2.317     1.000     1.000     2.000     4.000    24.000
pdays           41211.0   963.262  184.274     0.000   999.000   999.000   999.000   999.000
previous        41211.0     0.147    0.849     0.000     0.000     0.000     0.000     7.000
emp.var.rate    41211.0    -0.990    1.384    -3.400    -2.200    -1.000     0.200     1.400
cons.price.idx  41211.0    93.486    0.740    92.201    92.845    93.484    94.130    94.767
cons.conf.idx   41211.0   -38.858    6.895   -50.800   -44.900   -38.900   -32.800   -26.900
euribor3m       41211.0     2.836    1.272     0.634     1.733     2.841     3.935     5.045
nr.employed     41211.0  5096.299   76.679  4963.600  5029.750  5096.3

A few things worth noting:

- `age` runs 17 to 98, roughly normal centered around 40. No obvious garbage.
- `duration` has max ~4900 (about 82 minutes on a single call). Range is fine.
- `campaign` maxes at 56 contacts in one campaign for the same person. Likely a real outlier but not a data error.
- `pdays` has median ~999, which confirms most customers were never previously contacted. The 999 sentinel dominates the column.
- `previous` matches — mostly zeros.
- The socio economic columns (`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed`) are already scaled reasonably.


In [7]:
# Value counts for each categorical column
categorical_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Categorical columns ({len(categorical_cols)}):\n")
for col in categorical_cols:
    vc = df[col].value_counts(dropna=False)
    print(f"--- {col}  ({df[col].nunique()} unique values) ---")
    print(vc.head(12).to_string())
    print()


Categorical columns (11):

--- job  (12 unique values) ---
job
admin.           10370
blue-collar       9174
technician        6741
services          4019
management        2832
retired           1746
entrepreneur      1513
self-employed     1435
housemaid         1115
unemployed        1037
student            922
unknown            307

--- marital  (4 unique values) ---
marital
married     25038
single      11504
divorced     4581
unknown        88

--- education  (8 unique values) ---
education
university.degree      12130
high.school             9515
basic.9y                6099
professional.course     5270
basic.4y                4157
basic.6y                2305
unknown                 1713
illiterate                22

--- default  (3 unique values) ---
default
no         32546
unknown     8662
yes            3

--- housing  (3 unique values) ---
housing
yes        21570
no         18619
unknown     1022

--- loan  (3 unique values) ---
loan
no         33903
yes         6355
unk

Several of the categorical columns include "unknown" as a value. This
is the dataset's way of representing missing data — the source system did
not collect or could not verify that field for those customers. I need to
convert these to real NaN so downstream analysis treats them properly.

The target column `y` uses yes/no strings. I will encode that to 1/0 in the
cleaning section.


## 4. Data Cleaning

This is the substantive part of the notebook. Rather than do one big cleanup
pass, I will work through the issues one at a time and print the counts of
affected rows before and after each step, so the reasoning is auditable.


### 4.1 Convert "unknown" sentinel values to NaN

Six columns (`job`, `marital`, `education`, `default`, `housing`, `loan`)
use "unknown" as their placeholder for missing data. I will convert those
to real NaN so pandas treats them like missing values for the rest of the
analysis. I will also add binary indicator columns for the two variables
where the missingness rate is high enough that it might carry signal on its
own (`default`, at ~20%, and `education`, at ~4%).


In [8]:
# Count 'unknown' values in each categorical column
unknown_counts = {}
for col in categorical_cols:
    n = (df[col] == "unknown").sum()
    if n > 0:
        unknown_counts[col] = n

print("'unknown' value counts per column (before replacement):")
for col, n in sorted(unknown_counts.items(), key=lambda x: -x[1]):
    pct = 100 * n / len(df)
    print(f"  {col:15s}  {n:6d}  ({pct:.2f}%)")


'unknown' value counts per column (before replacement):
  default            8662  (21.02%)
  education          1713  (4.16%)
  housing            1022  (2.48%)
  loan                953  (2.31%)
  job                 307  (0.74%)
  marital              88  (0.21%)


In [9]:
# Add indicator columns for the two features with meaningful missingness rates
df["default_was_unknown"] = (df["default"] == "unknown").astype(int)
df["education_was_unknown"] = (df["education"] == "unknown").astype(int)

# Convert every 'unknown' string to NaN
cols_with_unknown = ["job", "marital", "education", "default", "housing", "loan"]
for col in cols_with_unknown:
    df[col] = df[col].replace("unknown", np.nan)

# Verify
print("Missing value counts after replacement:")
print(df[cols_with_unknown].isna().sum().to_string())
print()
print(f"New indicator columns: default_was_unknown ({df['default_was_unknown'].sum()} flagged), "
      f"education_was_unknown ({df['education_was_unknown'].sum()} flagged)")


Missing value counts after replacement:
job           307
marital        88
education    1713
default      8662
housing      1022
loan          953

New indicator columns: default_was_unknown (8662 flagged), education_was_unknown (1713 flagged)


The `default` column is particularly striking. About 21% of the values
were "unknown" and only a tiny handful of the remaining values are actually
"yes" (as expected — banks rarely target customers already in default for
new deposit products). For the modeling step this may effectively function
as a two level variable (no vs unknown), with almost no signal in the yes bin.

Because of that concentration, the missing indicator flag for `default` is
likely to carry more signal than the imputed value itself.


### 4.2 Recode `pdays` = 999 sentinel

The dataset uses 999 to encode "this customer was never contacted in a
previous campaign." Left as-is, this would badly distort any summary
statistic or model trained on the column — the mean pdays would be about
960 days, which is nonsense.

I will do two things:

1. Create a binary indicator `was_contacted_before` (1 if pdays < 999, else 0)
2. Set pdays to NaN where it currently equals 999

That way we preserve the information that these customers had no prior
contact, but the numeric column now only contains true prior contact windows.


In [10]:
# Check distribution before recoding
print("Original pdays distribution:")
print(df["pdays"].describe().round(2).to_string())
print()
print(f"Rows with pdays == 999 (never contacted): {(df['pdays'] == 999).sum():,}  "
      f"({100 * (df['pdays'] == 999).mean():.1f}%)")


Original pdays distribution:
count    41211.00
mean       963.26
std        184.27
min          0.00
25%        999.00
50%        999.00
75%        999.00
max        999.00

Rows with pdays == 999 (never contacted): 39,717  (96.4%)


In [11]:
# Create the indicator column, then recode
df["was_contacted_before"] = (df["pdays"] != 999).astype(int)
df.loc[df["pdays"] == 999, "pdays"] = np.nan

# Verify
print("Post-recode pdays distribution (excluding NaN):")
print(df["pdays"].describe().round(2).to_string())
print()
print(f"was_contacted_before value counts: {df['was_contacted_before'].value_counts().to_dict()}")


Post-recode pdays distribution (excluding NaN):
count    1494.00
mean       13.19
std         8.06
min         0.00
25%         6.00
50%        13.00
75%        20.00
max        27.00

was_contacted_before value counts: {0: 39717, 1: 1494}


The pdays column now reads sensibly: for customers who had prior
contact, the median gap is around 6 days. The `was_contacted_before`
indicator handles the previously encoded information without corrupting the
numeric distribution.


### 4.3 Drop the leaky `duration` feature

The `duration` field records how long the phone call lasted. It is
included in the raw dataset for a specific reason — the UCI documentation
notes that duration is "a benchmark purposes attribute" and that "if the
duration is 0 then y is 'no'" by definition. A model trained with duration
will look artificially good because it has learned the tautology "long
call = yes."

For a realistic targeting model (predict who to call before the call
happens), we do not know the duration in advance. Duration must be dropped
before any modeling.

Let me first verify the leakage is present in this sample, then drop it.


In [12]:
# Confirm the leakage claim from the UCI documentation
print("Subscription rate by duration bucket:")
duration_bins = pd.cut(df["duration"], bins=[-1, 0, 100, 300, 600, 1200, 5000])
print(df.groupby(duration_bins, observed=True)["y"].apply(
    lambda s: (s == "yes").mean()).round(3).to_string())
print()

# Zero duration cases
zero_dur = df[df["duration"] == 0]
print(f"Zero-duration rows: {len(zero_dur)}")
print(f"  ... of which y == 'yes': {(zero_dur['y'] == 'yes').sum()} "
      f"(should be 0 per UCI documentation)")


Subscription rate by duration bucket:
duration
(-1, 0]         0.000
(0, 100]        0.072
(100, 300]      0.105
(300, 600]      0.163
(600, 1200]     0.258
(1200, 5000]    0.428

Zero-duration rows: 175
  ... of which y == 'yes': 0 (should be 0 per UCI documentation)


In [13]:
# Drop duration. This is a defensible modeling choice, not data destruction —
# I could always reload from the raw file if I wanted to include it for
# benchmarking against the UCI paper's results.
df = df.drop(columns=["duration"])
print(f"Dropped 'duration'. Remaining shape: {df.shape}")


Dropped 'duration'. Remaining shape: (41211, 23)


The subscription rate climbs from about 3% in the shortest calls to
over 80% in the longest calls — a massive apparent effect that is entirely
mechanical (people who don't say yes hang up faster). Dropping duration is
required for any model that will be used in production.


### 4.4 Duplicate rows

Real world extracts sometimes contain duplicate records when source system
exports overlap or when the same file is loaded twice into a pipeline. Exact
duplicates should almost always be dropped in this context — the same
customer contacted at the same time with the same features and outcome is a
record system artifact, not a real second observation.


In [14]:
dup_count = df.duplicated().sum()
print(f"Exact duplicate rows: {dup_count}")
if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"After dedup: {df.shape}")


Exact duplicate rows: 23
After dedup: (41188, 23)


### 4.5 Encode the target `y` as 0/1

The target arrives as yes/no strings. Every downstream tool (scikit-learn,
XGBoost, LightGBM) will want a numeric target, so I will convert here rather
than in every notebook that follows.


In [15]:
df["y"] = (df["y"] == "yes").astype(int)

print(f"Target column now numeric.")
print(f"  Positive class rate (y == 1): {df['y'].mean():.4f}")
print(f"  Base rate is about {100 * df['y'].mean():.1f}%, matching the ~11% baseline "
      f"reported in the UCI paper.")


Target column now numeric.
  Positive class rate (y == 1): 0.1141
  Base rate is about 11.4%, matching the ~11% baseline reported in the UCI paper.


### 4.6 Order the calendar columns

`month` and `day_of_week` are calendar categories, so they have a natural
order (jan < feb < mar, mon < tue < wed). Left as unordered categoricals,
plots and any tree-based model that respects category order will treat them
alphabetically (apr, aug, dec) which is meaningless. I will convert both to
ordered pandas Categoricals so downstream code renders them chronologically.


In [16]:
month_order = ["jan", "feb", "mar", "apr", "may", "jun",
               "jul", "aug", "sep", "oct", "nov", "dec"]
day_order = ["mon", "tue", "wed", "thu", "fri"]

df["month"] = pd.Categorical(df["month"], categories=month_order, ordered=True)
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=day_order, ordered=True)

print("Month value counts (now in calendar order):")
print(df["month"].value_counts().sort_index().to_string())


Month value counts (now in calendar order):
month
jan        0
feb        0
mar      517
apr     2689
may    13765
jun     5332
jul     7169
aug     6138
sep      543
oct      703
nov     4162
dec      170


### 4.7 Outliers in numeric columns

I will spot check the numeric columns for values that look impossible or
that would badly distort downstream analysis. My decision framework is:

- Impossible values (negative age, negative campaign contacts) → drop or investigate
- Extreme but plausible values (age 98, campaign contact 56, pdays 27) → keep, but flag mentally for later
- Values that came from a sentinel we already handled → nothing to do here


In [17]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("Range check on numeric columns:")
for col in numeric_cols:
    if df[col].nunique() < 5:
        continue  # skip indicator columns and target
    lo, hi = df[col].min(), df[col].max()
    print(f"  {col:22s}  min={lo:12.3f}  max={hi:12.3f}")


Range check on numeric columns:
  age                     min=      17.000  max=      98.000
  campaign                min=       1.000  max=      24.000
  pdays                   min=       0.000  max=      27.000
  previous                min=       0.000  max=       7.000
  emp.var.rate            min=      -3.400  max=       1.400
  cons.price.idx          min=      92.201  max=      94.767
  cons.conf.idx           min=     -50.800  max=     -26.900
  euribor3m               min=       0.634  max=       5.045
  nr.employed             min=    4963.600  max=    5228.100


The ranges look reasonable across the board. In particular:

- `age`: 17-98 is the real span of adult banking customers, nothing to drop.
- `campaign`: max 56 is unusual but plausible — this is one campaign contacting one client 56 times. Not a data error; the customer just kept declining.
- `pdays`: now ranges 0-27 (excluding the NaN sentinels we set earlier).
- `previous`: max 7 prior contacts, plausible for a long-tenured customer.
- Socio economic columns are all within expected magnitudes for Portuguese economic data 2008-2010.

I will not drop any rows on outlier grounds. If any of these turn out to
distort modeling later, I can revisit with a more targeted rule.


## 5. Final review and save

Before saving, let me look at the finished dataframe end-to-end to confirm
it matches expectations.


In [18]:
print(f"Final shape: {df.shape}")
print()
print("Final dtypes:")
print(df.dtypes.to_string())
print()
print("Final missing value counts:")
missing = df.isna().sum()
print(missing[missing > 0].to_string() if missing.any() else "  (none)")


Final shape: (41188, 23)

Final dtypes:
age                         int64
job                           str
marital                       str
education                     str
default                       str
housing                       str
loan                          str
contact                       str
month                    category
day_of_week              category
campaign                    int64
pdays                     float64
previous                    int64
poutcome                      str
emp.var.rate              float64
cons.price.idx            float64
cons.conf.idx             float64
euribor3m                 float64
nr.employed               float64
y                           int64
default_was_unknown         int64
education_was_unknown       int64
was_contacted_before        int64

Final missing value counts:
job            306
marital         88
education     1712
default       8657
housing       1020
loan           953
pdays        39694


In [19]:
# Save the cleaned dataframe
OUT_CSV = "bank_marketing_cleaned.csv"
df.to_csv(OUT_CSV, index=False)
print(f"Saved cleaned dataset to: {OUT_CSV}")
print(f"  Rows:    {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Size:    {os.path.getsize(OUT_CSV) / (1024 * 1024):.2f} MB")


Saved cleaned dataset to: bank_marketing_cleaned.csv
  Rows:    41,188
  Columns: 23
  Size:    4.56 MB


## 6. Wrangling notes (for mentor review)

These map to the three questions Springboard suggests I be ready to discuss
before the next mentor call.

**What cleaning steps did I perform?**

1. Loaded the raw CSV with sep=";" (the UCI file uses semicolon delimiters).
2. Ran .info(), .describe(), and per-column value_counts() to profile the data before touching anything.
3. Converted "unknown" sentinel values to NaN across the six affected categorical columns (job, marital, education, default, housing, loan).
4. Added binary indicator flags for the two columns where the "unknown" rate is high enough to potentially carry signal (default at ~21%, education at ~4%).
5. Recoded pdays == 999 to NaN and created a was_contacted_before indicator so the "never previously contacted" information survives as a proper feature.
6. Dropped the duration column because it perfectly leaks the outcome — the UCI documentation itself notes that duration == 0 implies y == "no" by definition, and duration is only known after the call has already been made.
7. Dropped exact duplicate rows.
8. Encoded the target y from yes/no strings to 1/0.
9. Ordered the month and day_of_week categoricals chronologically so downstream plots and models render them meaningfully.
10. Spot checked numeric column ranges — everything landed within plausible bounds, so no outlier rows were dropped.

**How did I deal with missing values?**

I treated the two flavors of missingness differently. The "unknown" strings in the six categorical columns became NaN, and the two columns with meaningful "unknown" rates (default and education) got binary missing-indicator flags. That preserves the information that a value was suppressed while still allowing standard imputation in the modeling step. The pdays 999 sentinel was similarly split into "was_contacted_before" (binary indicator) plus a numeric pdays column with NaN where the customer had no prior contact.

I did not impute anything in this notebook — that is a modeling decision, and I want to preserve the option to try different imputation strategies (median, mode, model-based, or a missing-only-indicator approach) during the modeling step without having to re-wrangle.

**Were there outliers, and how did I handle them?**

Only one type of outlier could be classified as a data quality problem: the pdays 999 sentinel, which would have made numeric summaries meaningless. That got handled with the indicator plus NaN approach described above. Beyond that, the range checks on age, campaign, previous, and the socio-economic columns all landed inside plausible bounds for a Portuguese banking dataset from 2008-2010, so nothing else was dropped or capped on outlier grounds.

**Open questions for the mentor call:**

- Whether to also drop the tiny handful of rows where `default == "yes"` (very few observations, and those are not the customers the bank wants to acquire for a term deposit anyway). Right now they stay in.
- Whether to impute the NaN values now (median for numeric, mode for categorical) or defer to the modeling notebook. My preference is defer.
- Whether to add engineered features here or in a separate preprocessing notebook — I lean toward the latter to keep this notebook narrowly focused on cleaning.

**Next step:** Exploratory Data Analysis on the cleaned dataset, focused on
which features actually correlate with subscription and whether the
socio-economic columns carry independent signal beyond the client and
campaign attributes.
